# Notebook 28 — Declining Oscillator: Necessary Conditions

**Original goal:** Find declining oscillator outside the cryosphere (Greenland ice mass, glacier area).

**Data access reality:** Greenland GRACE/GRACE-FO data requires authentication. PIOMAS sea ice volume URLs are 404. The only accessible glacier dataset is WGMS global mass balance — which is **annual** data (no monthly oscillation).

**Pivot:** This constraint is actually scientifically useful. Instead of just finding more examples, we can test the necessary conditions for declining oscillator class more rigorously:

- **Part A:** WGMS glacier mass balance (annual) — strong long-term decline but no annual oscillation. Does it land in declining_osc or trend? This tests whether the oscillation component is necessary.
- **Part B:** Synthetic parameter sweep — vary oscillation frequency and decline strength on a 2D grid. Map which combinations produce which class. Gives a phase diagram of the attractor basins.

**The phase diagram is the experiment.** It tells us: in 6-feature space, exactly where does 'declining oscillator' begin and end? What are the tipping points — amplitude of decline, number of cycles — that push a series across the boundary into trend, oscillator, or integrated_trend?

---

## Pre-run Predictions

| Test | Prediction | Reasoning |
|---|---|---|
| WGMS glacier → class | trend or integrated_trend | Annual data, no oscillation — only the decline is visible |
| Phase diagram: high decline + few cycles | trend class | Decline dominates; oscillation invisible |
| Phase diagram: low decline + many cycles | oscillator class | Oscillation dominates; decline too small to register |
| Declining oscillator basin | Narrow band | Both decline AND oscillation must be present at the right scale |
| Minimum cycles for declining_osc | 2–4 per window | Fewer than 2 cycles looks like trend; more than 4 looks like seasonal |


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import requests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

CLASS_NAMES = [
    'burst', 'eco_cycle', 'oscillator', 'seasonal',
    'trend', 'integrated_trend', 'irregular_osc', 'declining_osc'
]
N_CLASSES = len(CLASS_NAMES)
SEQ_LEN   = 64
SEED      = 42
rng       = np.random.default_rng(SEED)
CACHE     = Path('/tmp/nb28_cache')
CACHE.mkdir(exist_ok=True)

TIMEDOM_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']


def zscore(s):
    s = np.asarray(s, dtype=float)
    std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s - s.mean()


def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))


def extract_6f(s):
    arr = np.asarray(s, dtype=float)
    t   = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }


print('Helpers loaded.')

In [ ]:
# ============================================================
# Synthetic canonical class centroids in 6-feature space
# (reference points for nearest-class assignment)
# ============================================================

t64 = np.linspace(0, 1, SEQ_LEN)

GENERATORS = {
    'burst': lambda r: zscore(np.exp(-(t64 - r.uniform(0.15,0.50))**2 / (2*r.uniform(0.05,0.15)**2)) + r.normal(0, 0.05, SEQ_LEN)),
    'eco_cycle': lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,3.5)*t64) + 0.4*np.sin(4*np.pi*r.uniform(1.5,3.5)*t64) + r.normal(0, 0.12, SEQ_LEN)),
    'oscillator': lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64 + r.uniform(0,np.pi)) + r.normal(0, 0.05, SEQ_LEN)),
    'seasonal': lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64) + 0.25*np.sin(4*np.pi*r.uniform(3,6)*t64) + r.normal(0, 0.04, SEQ_LEN)),
    'trend': lambda r: zscore(t64 + r.uniform(0.05,0.30)*t64**2 + r.normal(0, 0.02, SEQ_LEN)),
    'integrated_trend': lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(0.015,0.035) + r.normal(0, 0.003, SEQ_LEN))),
    'irregular_osc': lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(0.3,0.8,SEQ_LEN)) + r.normal(0, 0.3, SEQ_LEN)) * 1.4),
    'declining_osc': lambda r: zscore(np.linspace(r.uniform(0.9,1.2), r.uniform(0.35,0.65), SEQ_LEN) * np.sin(2*np.pi*r.uniform(2.5,5.5)*t64) + np.linspace(0, r.uniform(-0.8,-0.4), SEQ_LEN) + r.normal(0, 0.05, SEQ_LEN)),
}

# Generate 200 instances per class, compute centroids
from sklearn.preprocessing import StandardScaler

all_records = []
for cls_name, gen_fn in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + list(GENERATORS).index(cls_name)*1000 + i)
        f = extract_6f(gen_fn(r))
        f['class'] = cls_name
        all_records.append(f)

df_ref = pd.DataFrame(all_records)
scaler = StandardScaler()
X_ref  = scaler.fit_transform(df_ref[TIMEDOM_COLS].values)

centroids = {}
for i, cls_name in enumerate(GENERATORS):
    mask = df_ref['class'] == cls_name
    centroids[cls_name] = X_ref[mask].mean(axis=0)


def nearest_class(feat_dict):
    x = scaler.transform([[ feat_dict[c] for c in TIMEDOM_COLS ]])[0]
    dists = {cls: np.linalg.norm(x - c) for cls, c in centroids.items()}
    return min(dists, key=dists.get), dists


print('Reference class centroids computed from 200 synthetic instances each.')
print('\n6-feature centroid profile (mean per class):')
print(df_ref.groupby('class')[TIMEDOM_COLS].mean().round(3).to_string())

In [ ]:
# ============================================================
# PART A — WGMS Glacier Mass Balance
# Annual data. Strong long-term decline. No monthly oscillation.
# Question: does decline alone produce declining_osc, or is
# the annual oscillation a necessary condition?
# ============================================================

dest = Path('/tmp/nb28_cache/wgms_mb.csv')
if not dest.exists():
    r = requests.get('https://wgms.ch/data/faq/mb_ref.csv',
                     headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    dest.write_bytes(r.content)

df_wgms = pd.read_csv(dest)
print('WGMS columns:', df_wgms.columns.tolist())
print(df_wgms.head(5).to_string())
print(f'\n{len(df_wgms)} years: {df_wgms["Year"].min()} – {df_wgms["Year"].max()}')

In [ ]:
# ============================================================
# A2: 6-feature analysis of WGMS data
# Use both: annual mass balance (REF_regionAVG) and
# cumulative mass balance (REF_regionAVG_cum-rel-1970)
# ============================================================

cum_col  = [c for c in df_wgms.columns if 'cum' in c][0]
ann_col  = 'REF_regionAVG'

wgms_cum = df_wgms[cum_col].values.astype(float)
wgms_ann = df_wgms[ann_col].values.astype(float)

# Normalise each as a 64-point series (resample to SEQ_LEN)
from scipy.interpolate import interp1d

def resample_64(series):
    x_old = np.linspace(0, 1, len(series))
    x_new = np.linspace(0, 1, SEQ_LEN)
    return interp1d(x_old, series, kind='linear')(x_new)

wgms_cum_64 = zscore(resample_64(wgms_cum))
wgms_ann_64 = zscore(resample_64(wgms_ann))

# 6-feature fingerprint
f_cum = extract_6f(wgms_cum_64)
f_ann = extract_6f(wgms_ann_64)

cls_cum, dists_cum = nearest_class(f_cum)
cls_ann, dists_ann = nearest_class(f_ann)

print('WGMS cumulative mass balance:')
print(f'  6-feature fingerprint: {f_cum}')
print(f'  Nearest class: {cls_cum}')
print(f'  Top-3 distances: {sorted(dists_cum.items(), key=lambda x: x[1])[:3]}')

print()
print('WGMS annual mass balance:')
print(f'  6-feature fingerprint: {f_ann}')
print(f'  Nearest class: {cls_ann}')
print(f'  Top-3 distances: {sorted(dists_ann.items(), key=lambda x: x[1])[:3]}')

# Compare to synthetic declining_osc fingerprint
mask_dec = df_ref['class'] == 'declining_osc'
dec_mean = df_ref[mask_dec][TIMEDOM_COLS].mean()
print()
print('Synthetic declining_osc mean features (for comparison):')
for col in TIMEDOM_COLS:
    print(f'  {col:20s}: declining_osc={dec_mean[col]:7.3f}  wgms_cum={f_cum[col]:7.3f}')

In [ ]:
# ============================================================
# A3: Visualise WGMS series vs synthetic declining_osc
# ============================================================

# Generate a canonical declining_osc for comparison
r_ref = np.random.default_rng(SEED)
dec_osc_ref = zscore(
    np.linspace(1.0, 0.5, SEQ_LEN) * np.sin(2*np.pi*3.5*t64) +
    np.linspace(0, -0.6, SEQ_LEN)
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(wgms_cum_64, color='steelblue')
axes[0].set_title(f'WGMS cumulative mass balance\n(resampled to 64 pts, z-scored)\nNearest class: {cls_cum}')

axes[1].plot(wgms_ann_64, color='coral')
axes[1].set_title(f'WGMS annual mass balance\n(resampled to 64 pts, z-scored)\nNearest class: {cls_ann}')

axes[2].plot(dec_osc_ref, color='green')
axes[2].set_title('Synthetic declining_osc (reference)\n3.5 cycles, amplitude 1.0→0.5\nmean shift 0→-0.6')

for ax in axes:
    ax.set_xlabel('Time step (normalised)'); ax.set_ylabel('z-score')

plt.suptitle('Part A — WGMS Glacier Mass Balance vs Declining Oscillator', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PART B — Phase diagram: oscillation × decline
# Vary n_cycles (0.5 to 7) and decline_strength (0 to 1.5)
# For each combination, classify 20 synthetic instances.
# Map: which (cycles, decline) combinations produce which class?
# ============================================================

N_CYCLES_VALS   = np.linspace(0.5, 7.0, 20)   # oscillation cycles per window
DECLINE_VALS    = np.linspace(0.0, 1.5, 20)   # amplitude decline (1.0 → 1-decline)
N_INSTANCES     = 20                           # instances per (cycles, decline) pair

phase_results = []  # (n_cycles, decline, dominant_class, purity)

for n_cyc in N_CYCLES_VALS:
    for dec in DECLINE_VALS:
        votes = []
        for i in range(N_INSTANCES):
            r = np.random.default_rng(SEED + int(n_cyc*100) + int(dec*100) + i)
            freq      = 2 * np.pi * n_cyc
            amp_end   = max(0.1, 1.0 - dec)
            amplitude = np.linspace(1.0, amp_end, SEQ_LEN)
            mean_shift = np.linspace(0, -dec * 0.5, SEQ_LEN)
            noise     = r.normal(0, 0.05, SEQ_LEN)
            s = zscore(amplitude * np.sin(freq * t64) + mean_shift + noise)
            cls, _ = nearest_class(extract_6f(s))
            votes.append(cls)

        from collections import Counter
        counts   = Counter(votes)
        dom_cls  = counts.most_common(1)[0][0]
        purity   = counts[dom_cls] / N_INSTANCES
        phase_results.append({'n_cycles': n_cyc, 'decline': dec,
                               'class': dom_cls, 'purity': purity})

df_phase = pd.DataFrame(phase_results)
print(f'Phase diagram: {len(df_phase)} (n_cycles × decline) combinations')
print(f'Class distribution:\n{df_phase["class"].value_counts().to_string()}')

In [ ]:
# ============================================================
# B2: Plot the phase diagram
# ============================================================

CLASS_COLORS = {
    'burst': '#e41a1c',         'eco_cycle': '#ff7f00',
    'oscillator': '#4daf4a',    'seasonal': '#984ea3',
    'trend': '#a65628',         'integrated_trend': '#f781bf',
    'irregular_osc': '#999999', 'declining_osc': '#377eb8',
}

n_cyc_grid = N_CYCLES_VALS
dec_grid   = DECLINE_VALS

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: dominant class
ax = axes[0]
for _, row in df_phase.iterrows():
    color = CLASS_COLORS.get(row['class'], 'black')
    alpha = 0.4 + 0.6 * row['purity']
    ax.scatter(row['n_cycles'], row['decline'],
               c=color, s=180, alpha=alpha, marker='s', edgecolors='none')

# Legend
from matplotlib.patches import Patch
seen = df_phase['class'].unique()
legend_els = [Patch(facecolor=CLASS_COLORS[c], label=c) for c in CLASS_NAMES if c in seen]
ax.legend(handles=legend_els, loc='upper right', fontsize=8, framealpha=0.9)

# Mark the real arctic/antarctic sea ice operating point
ax.axvline(x=5.3, color='navy', linestyle='--', linewidth=1.2, alpha=0.7,
           label='Arctic ice (5.3 cycles/window)')
ax.set_xlabel('Oscillation cycles per window')
ax.set_ylabel('Amplitude decline (1.0 → 1-decline)')
ax.set_title('Phase diagram — dominant shape class\n(colour = class, opacity = purity)')
ax.annotate('Arctic/Antarctic\nsea ice zone', xy=(5.3, 0.75),
            xytext=(4.0, 1.2), fontsize=8, color='navy',
            arrowprops=dict(arrowstyle='->', color='navy'))

# Right: purity heatmap
ax = axes[1]
purity_grid = df_phase.pivot(index='decline', columns='n_cycles', values='purity')
sns.heatmap(purity_grid.iloc[::-1], ax=ax, cmap='YlOrRd_r', vmin=0.3, vmax=1.0,
            cbar_kws={'label': 'Purity (fraction dominant class)'},
            xticklabels=[f'{v:.1f}' if i%4==0 else '' for i,v in enumerate(N_CYCLES_VALS)],
            yticklabels=[f'{v:.2f}' if i%4==0 else '' for i,v in enumerate(DECLINE_VALS[::-1])])
ax.set_xlabel('Oscillation cycles per window')
ax.set_ylabel('Amplitude decline')
ax.set_title('Phase diagram — class assignment purity\n(dark = ambiguous boundary zone)')

plt.suptitle('Part B — Shape class phase diagram: oscillation × decline', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# B3: Extract the declining_osc basin boundaries
# ============================================================

dec_osc_zone = df_phase[df_phase['class'] == 'declining_osc']

print('Declining oscillator basin:')
if len(dec_osc_zone) > 0:
    print(f'  n_cycles range: {dec_osc_zone["n_cycles"].min():.2f} – {dec_osc_zone["n_cycles"].max():.2f}')
    print(f'  decline range:  {dec_osc_zone["decline"].min():.2f} – {dec_osc_zone["decline"].max():.2f}')
    print(f'  n cells in basin: {len(dec_osc_zone)} / {len(df_phase)} ({len(dec_osc_zone)/len(df_phase)*100:.1f}%)')
    print(f'  mean purity: {dec_osc_zone["purity"].mean():.3f}')
else:
    print('  No cells assigned to declining_osc — check centroid definition.')

print()
print('Boundary classes (what does declining_osc border?)')
print(df_phase.groupby('class')[['n_cycles', 'decline']].agg(['min','max']).round(2).to_string())

print()
print('Class assignment by n_cycles (averaged over all decline values):')
print(df_phase.groupby('n_cycles')['class'].agg(lambda x: x.value_counts().index[0]).to_string())

print()
print('Class assignment by decline (averaged over all n_cycles values):')
print(df_phase.groupby('decline')['class'].agg(lambda x: x.value_counts().index[0]).to_string())

---
## Findings — Notebook 28

### Finding 73: [WGMS glacier mass balance — does decline without oscillation land in declining_osc?]

**Prediction:** trend or integrated_trend (annual data, no oscillation).

**Result:** *(run to find out)*

---

### Finding 74: [Phase diagram — what is the declining oscillator basin?]

**Prediction:** Narrow band requiring both sufficient oscillation (>2 cycles) AND sufficient decline.

**Result:** *(run to find out)*

---

### Finding 75: [Minimum oscillation and decline thresholds for declining_osc class]

**Prediction:** Minimum ~2 cycles/window, minimum decline ~0.3 amplitude units.

**Result:** *(run to find out)*
